In [1]:
!pip install -U ultralytics
!pip install kagglehub
!pip install opencv-python
!pip install gtts playsound

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 61.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 9.8 MB/s eta 0:00:00
  Created wheel for playsound: filename=playsound-1.3.0-py3-none-any.whl size=7020 sha256=a4078aff5dc41f514b9e3d934ea6bde0fe844216ac9621a340101a800292ccbe
  Stored in directory: /root/.cache/pip/wheels/cf/42/ff/7c587bae55eec67b909ca316b250d9b4daedbf272a3cbeb907
Successfully built playsound
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1


In [2]:
import kagglehub
import os

# Download dataset menggunakan KaggleHub
path = kagglehub.dataset_download("agungmrf/indonesian-sign-language-bisindo")

print("Path to dataset files:", path)
# Path ini kemungkinan akan mengarah ke './datasets/agungmrf/indonesian-sign-language-bisindo'
DATASET_PATH = str(path)


# Membuat file konfigurasi YAML untuk YOLOv8
yaml_content = f"""
# Konfigurasi Dataset BISINDO untuk YOLOv8
path: {DATASET_PATH}/bisindo  # Sesuaikan path ke folder utama 'bisindo'
train: images/train  # Asumsi ada sub-folder train/val di dalam images/labels
val: images/val
test: images/test # Opsional

# Jumlah kelas
nc: 26 # Asumsi 26 abjad A-Z

# Nama-nama kelas (Harus sesuai urutan index di file label .txt)
names: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
"""

# Simpan konfigurasi ke file YAML
with open('bisindo_data.yaml', 'w') as f:
    f.write(yaml_content)

print("File konfigurasi bisindo_data.yaml telah dibuat.")

100%|██████████| 1.36G/1.36G [01:05<00:00, 22.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/agungmrf/indonesian-sign-language-bisindo/versions/1
File konfigurasi bisindo_data.yaml telah dibuat.


In [3]:
from ultralytics import YOLO

# Muat model YOLOv8 (pilih versi, misal 'n' for nano, 's' for small)
# Nano adalah yang paling ringan dan tercepat untuk aplikasi real-time.
model = YOLO('yolov8n.pt')

# Latih model
# 'data': file konfigurasi YAML yang kita buat
# 'epochs': jumlah iterasi pelatihan (gunakan 5-10 untuk uji coba)
# 'imgsz': ukuran gambar input (sesuaikan, 640 adalah standar)
results = model.train(data='bisindo_data.yaml', epochs=30, imgsz=640, name='bisindo_yolov8_model')

# Simpan model terbaik setelah pelatihan
# Model terbaik biasanya tersimpan di runs/detect/bisindo_yolov8_model/weights/best.pt

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bisindo_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=

In [4]:
import shutil
import os
from google.colab import files

# --- 1. Definisikan Lokasi Hasil ---
# Folder ini adalah tempat YOLOv8 menyimpan semua hasil (weights, plots, logs)
OUTPUT_DIR = 'runs'
ZIP_FILENAME = 'yolov8_bisindo_results.zip'

print(f"Memulai kompresi folder hasil: {OUTPUT_DIR}...")

# --- 2. Kompresi Folder Hasil Menjadi File ZIP ---
try:
    # Menggunakan shutil.make_archive untuk membuat file ZIP
    # shutil.make_archive(nama_file_zip_tanpa_ekstensi, format, folder_yang_dikompres)
    shutil.make_archive(ZIP_FILENAME.replace('.zip', ''), 'zip', OUTPUT_DIR)
    print(f"Kompresi berhasil. File ZIP dibuat: {ZIP_FILENAME}")
except FileNotFoundError:
    print(f"ERROR: Folder hasil '{OUTPUT_DIR}' tidak ditemukan. Pastikan Anda sudah menjalankan pelatihan YOLOv8 di Colab ini.")
    # Exit skrip jika folder tidak ada
    # sys.exit() # Tidak perlu sys.exit di Colab

# --- 3. Unduh File ZIP ke Komputer Lokal ---
if os.path.exists(ZIP_FILENAME):
    try:
        print(f"Mengunduh file {ZIP_FILENAME}...")
        files.download(ZIP_FILENAME)
        print("Pengunduhan dimulai. Periksa jendela browser Anda.")
    except Exception as e:
        print(f"Gagal saat proses unduh: {e}")
else:
    print("File ZIP tidak ditemukan untuk diunduh.")

Memulai kompresi folder hasil: runs...
Kompresi berhasil. File ZIP dibuat: yolov8_bisindo_results.zip
Mengunduh file yolov8_bisindo_results.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Pengunduhan dimulai. Periksa jendela browser Anda.
